In [ ]:
import os
import glob
import zipfile
from google.colab import drive

# ── MOUNT GOOGLE DRIVE ─────────────────────────────────────────────
drive.mount('/content/drive/')

NUM_CLASS = 1  # change this (1, 2, 3...)
CLASS_NAME = f"class_{NUM_CLASS:02d}"


# ── BASE_DIR ─────────────────────────────────────────────────────

BASE_DIR = f'/content/drive/MyDrive/dataset_ad/{CLASS_NAME}/{CLASS_NAME}'
print("BASE_DIR =", BASE_DIR)

# ── PATHS ───────────────────────────────────────────────────────
TRAIN_GOOD_DIR = os.path.join(BASE_DIR, 'train', 'good')
TEST_ANOM_DIRS = glob.glob(os.path.join(BASE_DIR, 'train', 'anomaly_*'))
GT_ANOM_DIRS   = glob.glob(os.path.join(BASE_DIR, 'ground_truth_train', 'anomaly_*'))

print("TRAIN_GOOD_DIR =", TRAIN_GOOD_DIR)
print("TEST_ANOM_DIRS =", TEST_ANOM_DIRS)
print("GT_ANOM_DIRS =", GT_ANOM_DIRS)

# ── MODEL / DINOv2 ──────────────────────────────────────────────
DINO_MODEL = 'vit_base_patch14_dinov2.lvd142m'  # or vit_small / vit_large
FEATURE_LAYERS_HL = [6, 9, 12]   # alto livello
FEATURE_LAYERS_LL = [2, 4]       # basso livello

# ── IMAGE ───────────────────────────────────────────────────────
IMG_SIZE = 224
BATCH_SIZE = 16

# ── PATCHCORE ───────────────
CORESET_RATIO = 0.05
K_NEIGHBOURS = 1


# ── OUTPUT ──────────────────────────────────────────────────────
OUTPUT_DIR = f'/content/patchcore_output/{CLASS_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
# ── Install / upgrade dependencies ──────────────────────────────────────────
!pip install -q timm einops scikit-learn tqdm matplotlib opencv-python-headless

import os, glob, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

denorm = transforms.Compose([
    transforms.Normalize(mean=[-m/s for m, s in zip(mean, std)],
                         std=[1/s for s in std])
])


class ImageFolderFlat(Dataset):
    EXTS = {'.png'}

    def __init__(self, items, label, transform=None):
        if isinstance(items, str):
            items = [items]
        self.paths = []
        for item in items:
            if os.path.isdir(item):
                for p in glob.glob(os.path.join(item, '**', '*'), recursive=True):
                    if os.path.splitext(p)[1].lower() in self.EXTS:
                        self.paths.append(p)
            elif os.path.isfile(item):
                if os.path.splitext(item)[1].lower() in self.EXTS:
                    self.paths.append(item)
        self.paths = sorted(self.paths)
        self.label = label
        self.transform = transform
        print(f'Found {len(self.paths)} images | label={label}')

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.label, self.paths[idx]
from collections import defaultdict
from sklearn.model_selection import train_test_split

all_good = glob.glob(os.path.join(TRAIN_GOOD_DIR, '*.png'))

def get_obj_id(p):
    return os.path.basename(p).split('_view')[0]

groups = defaultdict(list)
for p in all_good:
    groups[get_obj_id(p)].append(p)

object_ids = list(groups.keys())

train_ids, val_ids = train_test_split(
    object_ids,
    test_size=0.2,
    random_state=42
)

train_paths = [p for oid in train_ids for p in groups[oid]]
val_good_paths = [p for oid in val_ids for p in groups[oid]]

print('Loading datasets...')

#ma mi serve dividere in test-train in fondo è una banca dati????
train_ds = ImageFolderFlat(train_paths, label=0, transform=transform)
test_good = ImageFolderFlat(val_good_paths, label=0, transform=transform)
test_anom = ImageFolderFlat(TEST_ANOM_DIRS, label=1, transform=transform)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_good_dl = DataLoader(test_good, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_anom_dl = DataLoader(test_anom, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import timm

class DINOv2PatchExtractor(nn.Module):
    """
    Wraps a DINOv2 ViT and returns multi-layer patch token features.
    Output shape per image: [N_patches, C_total]
    where C_total = len(FEATURE_LAYERS) * hidden_dim.
    """
    def __init__(self, model_name=DINO_MODEL, layers=FEATURE_LAYERS_HL):
        super().__init__()
        self.model = timm.create_model(
            model_name, pretrained=True, num_classes=0,
            dynamic_img_size=True
        )
        self.model.eval()
        self.layers  = layers
        self._hooks  = []
        self._feats  = {}
        self._register_hooks()

    def _register_hooks(self):
        for layer_idx in self.layers:
            block = self.model.blocks[layer_idx - 1]   # 1-indexed → 0-indexed
            def hook(module, inp, out, idx=layer_idx):
                # out: [B, N_tokens, D]  — first token is CLS, rest are patches
                self._feats[idx] = out[:, 1:, :]   # drop CLS
            h = block.register_forward_hook(hook)
            self._hooks.append(h)

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()

    @torch.no_grad()
    def forward(self, x):
        """Returns patch features: [B, N_patches, D_total]"""
        self._feats = {}
        _ = self.model(x)
        # concatenate features from all tapped layers
        feats = torch.cat([self._feats[l] for l in self.layers], dim=-1)  # [B, N, D*L]
        # L2-normalise per patch (improves NN distance quality)
        feats = nn.functional.normalize(feats, dim=-1)
        return feats


print(f'Loading {DINO_MODEL}...')

extractor_hl = DINOv2PatchExtractor(layers=FEATURE_LAYERS_HL).to(DEVICE)
extractor_ll = DINOv2PatchExtractor(layers=FEATURE_LAYERS_LL).to(DEVICE)

# Quick sanity check
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out_hl = extractor_hl(dummy)
out_ll = extractor_ll(dummy)

print(f'Patch feature HL shape: {out_hl.shape}')  # [B, N_patches, D_HL]
print(f'Patch feature LL shape: {out_ll.shape}')  # [B, N_patches, D_LL]

N_PATCHES = out_hl.shape[1]
FEAT_DIM_HL = out_hl.shape[2]
FEAT_DIM_LL = out_ll.shape[2]

In [ ]:
def build_memory_bank(dataloader, extractor, device, coreset_ratio):
    memory_bank_list = []
    for imgs, labels, paths in tqdm(dataloader, desc='Extracting & coreset'):
        imgs = imgs.to(device)
        feats = extractor(imgs)  # [B, N_patches, D]
        B, N, D = feats.shape
        feats = feats.reshape(-1, D).cpu()

        # Random coreset sul batch
        K = max(1, int(feats.shape[0] * coreset_ratio))
        idx = torch.randperm(feats.shape[0])[:K]
        memory_bank_list.append(feats[idx])

    memory_bank = torch.cat(memory_bank_list, dim=0)
    print(f'Memory bank final shape: {memory_bank.shape}')
    return memory_bank

In [ ]:
memory_bank_hl = build_memory_bank(train_dl, extractor_hl, DEVICE, CORESET_RATIO)
memory_bank_ll = build_memory_bank(train_dl, extractor_ll, DEVICE, CORESET_RATIO)

print(f'Memory bank HL shape: {memory_bank_hl.shape}')
print(f'Memory bank LL shape: {memory_bank_ll.shape}')


In [ ]:
import os
import torch

# Cartella di output su Google Drive
OUTPUT_DIR = f'/content/drive/MyDrive/patchcore_output/{CLASS_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

save_path = os.path.join(OUTPUT_DIR, "patchcore_dinov2_2layers.pt")

# Controllo se il file esiste già
if os.path.exists(save_path):
    raise FileExistsError(f"Il file {save_path} esiste già! Non verrà sovrascritto.")
else:
    torch.save({
        "memory_bank_HL"   : memory_bank_hl,
        "memory_bank_LL"   : memory_bank_ll,
        "feature_layers_HL": FEATURE_LAYERS_HL,
        "feature_layers_LL": FEATURE_LAYERS_LL
    }, save_path)
    print(f"File salvato correttamente in {save_path}")

In [ ]:
def compute_anomaly_scores(img_feats, memory_bank, k=1, batch_size=1024):
    P = img_feats.shape[0]
    patch_scores = torch.zeros(P)

    memory_bank = memory_bank.float()

    for start in range(0, P, batch_size):
        end = min(start + batch_size, P)
        q = img_feats[start:end].float()

        dists = torch.cdist(q, memory_bank)  # [bs, M]
        knn_dists = dists.topk(k, dim=1, largest=False).values
        patch_scores[start:end] = knn_dists.mean(dim=1)

    image_score = patch_scores.max().item()
    return patch_scores, image_score


def score_dataloader(dataloader, extractor, memory_bank, device):
    results = []

    for imgs, labels, paths in tqdm(dataloader, desc='Scoring'):
        imgs = imgs.to(device)
        with torch.no_grad():
            feats = extractor(imgs).cpu()  # [B, N_patches, D]
        for i in range(feats.shape[0]):
            ps, im_s = compute_anomaly_scores(feats[i], memory_bank)
            results.append({
                'path': paths[i],
                'label': labels[i].item(),
                'image_score': im_s,
                'patch_scores': ps
            })
    return results


In [ ]:
def patch_scores_to_heatmap(patch_scores, img_size):
    n = patch_scores.shape[0]
    side = int(n ** 0.5)
    patch_scores = patch_scores[:side * side]
    hmap = patch_scores.view(side, side).cpu().numpy()
    hmap = cv2.resize(hmap, (img_size, img_size), interpolation=cv2.INTER_LINEAR)    #cv2.INTER_LINEAR  cv2.INTER_NEAREST
    #hmap = (hmap - hmap.min()) / (hmap.max() - hmap.min() + 1e-8)
    return hmap



In [ ]:
import os
import torch

SAVE_PATH = f'/content/drive/MyDrive/patchcore_output/{CLASS_NAME}/patchcore_dinov2_2layers.pt'

# Caricamento
if not os.path.exists(SAVE_PATH):
    raise FileNotFoundError(f"Il file {SAVE_PATH} non esiste! Devi prima salvarlo.")

checkpoint = torch.load(SAVE_PATH, map_location=DEVICE)
memory_bank_hl = checkpoint['memory_bank_HL']
memory_bank_ll = checkpoint['memory_bank_LL']
FEATURE_LAYERS_HL = checkpoint['feature_layers_HL']
FEATURE_LAYERS_LL = checkpoint['feature_layers_LL']

print(f"File caricato correttamente da {SAVE_PATH}")

Da qui visualizzazione

In [ ]:
import numpy as np
from collections import deque

def extract_background(image, pad=20, threshold=10):
    # convert PIL → numpy se necessario
    if not isinstance(image, np.ndarray):
        image = np.array(image)
    h, w = image.shape[:2]
    gray = image if len(image.shape) == 2 else image.mean(axis=2)
    # seed mask (cornice)
    seed_mask = np.zeros((h, w), dtype=bool)
    seed_mask[:pad, :] = True
    seed_mask[-pad:, :] = True
    seed_mask[:, :pad] = True
    seed_mask[:, -pad:] = True
    seed_values = gray[seed_mask]
    bg_mean = seed_values.mean()
    visited = np.zeros((h, w), dtype=bool)
    bg_mask = np.zeros((h, w), dtype=bool)
    from collections import deque
    q = deque()
    ys, xs = np.where(seed_mask)
    for i, j in zip(ys, xs):
        q.append((i, j))
        visited[i, j] = True
        bg_mask[i, j] = True
    while q:
        i, j = q.popleft()
        for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
            ni, nj = i + di, j + dj
            if 0 <= ni < h and 0 <= nj < w and not visited[ni, nj]:
                if abs(gray[ni, nj] - bg_mean) < threshold:
                    visited[ni, nj] = True
                    bg_mask[ni, nj] = True
                    q.append((ni, nj))

    return bg_mask

import numpy as np
from scipy.ndimage import label as nd_label

def keep_largest_component(bg_mask):
    fg = (~bg_mask).astype(np.uint8)
    labeled, num = nd_label(fg)

    if num == 0:
        return np.zeros_like(bg_mask, dtype=bool)

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    largest_label = np.argmax(sizes)

    out = (labeled == largest_label)

    return ~out

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import average_precision_score
from math import ceil
from torchvision import transforms

# ── NOME FILE OUTPUT ─────────────────────────────────────────────
OUTPUT_FILE_NAME = "anomaly_visualization_HL_LL.png"
OUTPUT_PATH = f"/content/drive/MyDrive/patchcore_output/{OUTPUT_FILE_NAME}"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ── PARAMETRI ─────────────────────────────────────────────────────
IMGS_PER_ROW = 5
IMG_SIZE_PLOT = 224
ALPHA = 0.5

# ── FUNZIONE DENORMALIZZAZIONE ───────────────────────────────────
def denormalize_tensor(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=img_tensor.device)[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225], device=img_tensor.device)[:, None, None]
    img = img_tensor * std + mean
    img = torch.clamp(img, 0, 1)
    img = (img * 255).byte()
    return transforms.ToPILImage()(img.cpu())

# ── FUNZIONE HEATMAP ─────────────────────────────────────────────
def overlay_heatmap(img_pil, patch_scores, img_size=IMG_SIZE_PLOT, alpha=ALPHA, colormap=cv2.COLORMAP_JET):
    # patch_scores già in [0,1] o qualsiasi scala, non normalizziamo qui
    hmap = patch_scores_to_heatmap(patch_scores, img_size)
    heatmap_color = cv2.applyColorMap((hmap*255).astype(np.uint8), colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(np.array(img_pil.resize((img_size,img_size))), alpha, heatmap_color, 1-alpha, 0)
    return overlay, hmap

# ── PREPARAZIONE FIGURA ──────────────────────────────────────────
n_images = len(test_anom)
n_rows = ceil(n_images / IMGS_PER_ROW) * 3  # 3 righe per GT + HL + LL

fig, axs = plt.subplots(n_rows, IMGS_PER_ROW, figsize=(IMGS_PER_ROW*4, n_rows*4))
axs = np.array(axs)

for idx in range(n_images):
    img_tensor, label, path = test_anom[idx]
    img_pil = denormalize_tensor(img_tensor)
    img_to_plot = np.array(img_pil.resize((IMG_SIZE_PLOT, IMG_SIZE_PLOT)))

    # ── RIGHE E COLONNE ─────────────────────────────
    row_base = (idx // IMGS_PER_ROW) * 3
    col = idx % IMGS_PER_ROW

    # ── GROUND TRUTH SULL'IMMAGINE BASE ─────────────
    rel_dir = os.path.relpath(os.path.dirname(path), os.path.join(BASE_DIR, "train"))
    gt_mask_path = os.path.join(BASE_DIR, "ground_truth_train", rel_dir, os.path.basename(path))

    if os.path.exists(gt_mask_path):
        gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 0).astype(np.uint8)
        gt_mask_resized = cv2.resize(gt_mask, (IMG_SIZE_PLOT, IMG_SIZE_PLOT), interpolation=cv2.INTER_NEAREST)
        overlay_gt = img_to_plot.copy()
        overlay_gt[gt_mask_resized==1] = [255, 0, 0]
        img_base = cv2.addWeighted(img_to_plot, 0.7, overlay_gt, 0.3, 0)
    else:
        img_base = img_to_plot

    axs[row_base, col].imshow(img_base)
    axs[row_base, col].set_title(f"GT {idx}")
    axs[row_base, col].axis('off')

    # ── ESTRAI FEATURE HL/LL ─────────────────────────
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    feats_hl = extractor_hl(img_tensor)[0]
    feats_ll = extractor_ll(img_tensor)[0]

    if memory_bank_hl.device.type == "cpu":
        feats_hl = feats_hl.cpu()
        feats_ll = feats_ll.cpu()

    # ── PATCH SCORES HL/LL ──────────────────────────
    patch_scores_hl, _ = compute_anomaly_scores(feats_hl, memory_bank_hl)
    patch_scores_ll, _ = compute_anomaly_scores(feats_ll, memory_bank_ll)


    # ── HEATMAPS HL/LL (SENZA GT) ──────────────────
    overlay_hl, hmap_hl = overlay_heatmap(img_pil, patch_scores_hl)
    overlay_ll, hmap_ll = overlay_heatmap(img_pil, patch_scores_ll)

    # ── CALCOLO AP PIXEL-WISE ──────────────────────
    if os.path.exists(gt_mask_path):
        gt_resized = cv2.resize(gt_mask, (hmap_hl.shape[1], hmap_hl.shape[0]), interpolation=cv2.INTER_NEAREST)
        ap_hl = average_precision_score(gt_resized.flatten(), hmap_hl.flatten()) if gt_resized.sum()>0 else 0.0
        ap_ll = average_precision_score(gt_resized.flatten(), hmap_ll.flatten()) if gt_resized.sum()>0 else 0.0
    else:
        ap_hl = ap_ll = 0.0

    axs[row_base+1, col].imshow(overlay_hl)
    axs[row_base+1, col].set_title(f"HL heatmap\nAP: {ap_hl:.3f}")
    axs[row_base+1, col].axis('off')

    axs[row_base+2, col].imshow(overlay_ll)
    axs[row_base+2, col].set_title(f"LL heatmap\nAP: {ap_ll:.3f}")
    axs[row_base+2, col].axis('off')

# ── DISABILITA ASSI EXTRA ─────────────────────────
for r in range(n_rows):
    for c in range(IMGS_PER_ROW):
        if r*IMGS_PER_ROW + c >= n_images:
            axs[r, c].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=150)
print(f"Visualizzazione salvata in {OUTPUT_PATH}")
plt.show()

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import average_precision_score
from math import ceil
from torchvision import transforms

# ── NOME FILE OUTPUT ─────────────────────────────────────────────
OUTPUT_FILE_NAME = "anomaly_visualization_HL_LL.png"
OUTPUT_PATH = f"/content/drive/MyDrive/patchcore_output/{OUTPUT_FILE_NAME}"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# ── PARAMETRI ─────────────────────────────────────────────────────
IMGS_PER_ROW = 5
IMG_SIZE_PLOT = 224
ALPHA = 0.5

# ── FUNZIONE DENORMALIZZAZIONE ───────────────────────────────────
def denormalize_tensor(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=img_tensor.device)[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225], device=img_tensor.device)[:, None, None]
    img = img_tensor * std + mean
    img = torch.clamp(img, 0, 1)
    img = (img * 255).byte()
    return transforms.ToPILImage()(img.cpu())

# ── FUNZIONE HEATMAP ─────────────────────────────────────────────
def overlay_heatmap(img_pil, patch_scores, img_size=IMG_SIZE_PLOT, alpha=ALPHA, colormap=cv2.COLORMAP_JET):
    # patch_scores già in [0,1] o qualsiasi scala, non normalizziamo qui
    hmap = patch_scores_to_heatmap(patch_scores, img_size)
    heatmap_color = cv2.applyColorMap((hmap*255).astype(np.uint8), colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(np.array(img_pil.resize((img_size,img_size))), alpha, heatmap_color, 1-alpha, 0)
    return overlay, hmap

# ── PREPARAZIONE FIGURA ──────────────────────────────────────────
n_images = len(test_anom)
n_rows = ceil(n_images / IMGS_PER_ROW) * 3  # 3 righe per GT + HL + LL

fig, axs = plt.subplots(n_rows, IMGS_PER_ROW, figsize=(IMGS_PER_ROW*4, n_rows*4))
axs = np.array(axs)

for idx in range(n_images):
    img_tensor, label, path = test_anom[idx]
    img_pil = denormalize_tensor(img_tensor)
    img_to_plot = np.array(img_pil.resize((IMG_SIZE_PLOT, IMG_SIZE_PLOT)))

    # ── RIGHE E COLONNE ─────────────────────────────
    row_base = (idx // IMGS_PER_ROW) * 3
    col = idx % IMGS_PER_ROW

    # ── GROUND TRUTH SULL'IMMAGINE BASE ─────────────
    rel_dir = os.path.relpath(os.path.dirname(path), os.path.join(BASE_DIR, "train"))
    gt_mask_path = os.path.join(BASE_DIR, "ground_truth_train", rel_dir, os.path.basename(path))

    if os.path.exists(gt_mask_path):
        gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 0).astype(np.uint8)
        gt_mask_resized = cv2.resize(gt_mask, (IMG_SIZE_PLOT, IMG_SIZE_PLOT), interpolation=cv2.INTER_NEAREST)
        overlay_gt = img_to_plot.copy()
        overlay_gt[gt_mask_resized==1] = [255, 0, 0]
        img_base = cv2.addWeighted(img_to_plot, 0.7, overlay_gt, 0.3, 0)
    else:
        img_base = img_to_plot

    axs[row_base, col].imshow(img_base)
    axs[row_base, col].set_title(f"GT {idx}")
    axs[row_base, col].axis('off')

    # ── ESTRAI FEATURE HL/LL ─────────────────────────
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    feats_hl = extractor_hl(img_tensor)[0]
    feats_ll = extractor_ll(img_tensor)[0]

    if memory_bank_hl.device.type == "cpu":
        feats_hl = feats_hl.cpu()
        feats_ll = feats_ll.cpu()

    # ── PATCH SCORES HL/LL ──────────────────────────
    patch_scores_hl, _ = compute_anomaly_scores(feats_hl, memory_bank_hl)
    patch_scores_ll, _ = compute_anomaly_scores(feats_ll, memory_bank_ll)


    # ── HEATMAPS HL/LL (SENZA GT) ──────────────────
    overlay_hl, hmap_hl = overlay_heatmap(img_pil, patch_scores_hl)
    overlay_ll, hmap_ll = overlay_heatmap(img_pil, patch_scores_ll)

    # ── CALCOLO AP PIXEL-WISE ──────────────────────
    if os.path.exists(gt_mask_path):
        gt_resized = cv2.resize(gt_mask, (hmap_hl.shape[1], hmap_hl.shape[0]), interpolation=cv2.INTER_NEAREST)
        ap_hl = average_precision_score(gt_resized.flatten(), hmap_hl.flatten()) if gt_resized.sum()>0 else 0.0
        ap_ll = average_precision_score(gt_resized.flatten(), hmap_ll.flatten()) if gt_resized.sum()>0 else 0.0
    else:
        ap_hl = ap_ll = 0.0

    hl_max = float(np.max(hmap_hl))
    ll_max = float(np.max(hmap_ll))

    axs[row_base+1, col].imshow(overlay_hl)
    axs[row_base+1, col].set_title(
        f"HL heatmap\nAP: {ap_hl:.3f}\nMax: {hl_max:.3f}"
    )
    axs[row_base+1, col].axis('off')

    axs[row_base+2, col].imshow(overlay_ll)
    axs[row_base+2, col].set_title(
        f"LL heatmap\nAP: {ap_ll:.3f}\nMax: {ll_max:.3f}"
    )
    axs[row_base+2, col].axis('off')

# ── DISABILITA ASSI EXTRA ─────────────────────────
for r in range(n_rows):
    for c in range(IMGS_PER_ROW):
        if r*IMGS_PER_ROW + c >= n_images:
            axs[r, c].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=150)
print(f"Visualizzazione salvata in {OUTPUT_PATH}")
plt.show()

In [ ]:
import numpy as np
import cv2

def fuse_patch_heatmaps(hl_scores, hh_scores, img_size, low_thresh=0.6, high_thresh=0.5):
    # Assicurati che siano array 1D di patch
    n_low = hl_scores.shape[0]
    n_high = hh_scores.shape[0]
    side_low = int(n_low ** 0.5)
    side_high = int(n_high ** 0.5)

    # Applica soglie condizionali ai patch
    hl_masked = np.where(hl_scores > low_thresh, hl_scores, hl_scores*0.6)
    hh_masked = np.where(hh_scores > high_thresh, hh_scores, hh_scores*0.5)

    # AND-like patch per patch
    n = min(len(hl_masked), len(hh_masked))  # caso diverse dimensioni
    fused = hl_masked[:n] * hh_masked[:n]

    # Reshape a matrice quadrata e resize
    side = int(n ** 0.5)
    fused_map = fused[:side*side].reshape(side, side)
    heatmap_final = cv2.resize(fused_map, (img_size, img_size), interpolation=cv2.INTER_NEAREST)

    return heatmap_final

In [ ]:
#possibili idee se abbastanza alto per low_tresh fai un secondo low tresh più basso per i vicini ma devono essere attaccati massimo diagonale uno
def fuse_patch_heatmaps_max(
    hl_scores,
    hh_scores,
    img_size,
    low_thresh=0.8,
    high_thresh=0.5
):
    import numpy as np
    import cv2

    # Assicurati che siano array flat
    hl_scores = np.asarray(hl_scores).flatten()
    hh_scores = np.asarray(hh_scores).flatten()

    # Threshold soft
    hl_masked = np.where(
        hl_scores > low_thresh,
        hl_scores,
        hl_scores * 0.5
    )

    hh_masked = np.where(
        hh_scores > high_thresh,
        hh_scores,
        hh_scores * 0.5
    )

    # Allinea lunghezze
    n = min(len(hl_masked), len(hh_masked))

    # Fusion MAX invece di prodotto
    fused = np.maximum(
        hl_masked[:n],
        hh_masked[:n]
    )

    # Reshape quadrato
    side = int(n ** 0.5)
    fused_map = fused[:side * side].reshape(side, side)

    # Resize finale
    heatmap_final = cv2.resize(
        fused_map,
        (img_size, img_size),
        interpolation=cv2.INTER_NEAREST
    )

    return heatmap_final

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import average_precision_score
from math import ceil
from torchvision import transforms

OUTPUT_FILE_NAME = "anomaly_visualization_fused_2rows.png"
OUTPUT_PATH = f"/content/drive/MyDrive/patchcore_output/{OUTPUT_FILE_NAME}"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

IMGS_PER_ROW = 5
IMG_SIZE_PLOT = 224
ALPHA = 0.5

def denormalize_tensor(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=img_tensor.device)[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225], device=img_tensor.device)[:, None, None]
    img = img_tensor * std + mean
    img = torch.clamp(img, 0, 1)
    img = (img * 255).byte()
    return transforms.ToPILImage()(img.cpu())

def overlay_heatmap(img_pil, hmap, alpha=ALPHA, colormap=cv2.COLORMAP_JET):
    heatmap_color = cv2.applyColorMap((hmap*255).astype(np.uint8), colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(np.array(img_pil.resize((IMG_SIZE_PLOT, IMG_SIZE_PLOT))), alpha, heatmap_color, 1-alpha, 0)
    return overlay

# ── PREPARAZIONE FIGURA ──────────────────────────
n_images = len(test_anom)
n_rows = ceil(n_images / IMGS_PER_ROW) * 2  # base + fused heatmap

fig, axs = plt.subplots(n_rows, IMGS_PER_ROW, figsize=(IMGS_PER_ROW*4, n_rows*4))
axs = np.array(axs)

for idx in range(n_images):
    img_tensor, label, path = test_anom[idx]
    img_pil = denormalize_tensor(img_tensor)
    img_to_plot = np.array(img_pil.resize((IMG_SIZE_PLOT, IMG_SIZE_PLOT)))

    row_base = (idx // IMGS_PER_ROW) * 2
    col = idx % IMGS_PER_ROW

    # ── IMMAGINE BASE + GT ─────────────────────────
    rel_dir = os.path.relpath(os.path.dirname(path), os.path.join(BASE_DIR, "train"))
    gt_mask_path = os.path.join(BASE_DIR, "ground_truth_train", rel_dir, os.path.basename(path))

    if os.path.exists(gt_mask_path):
        gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 0).astype(np.uint8)
        gt_mask_resized = cv2.resize(gt_mask, (IMG_SIZE_PLOT, IMG_SIZE_PLOT), interpolation=cv2.INTER_NEAREST)
        overlay_gt = img_to_plot.copy()
        overlay_gt[gt_mask_resized==1] = [255, 0, 0]
        img_base = cv2.addWeighted(img_to_plot, 0.7, overlay_gt, 0.3, 0)
    else:
        img_base = img_to_plot

    axs[row_base, col].imshow(img_base)
    axs[row_base, col].set_title(f"GT {idx}")
    axs[row_base, col].axis('off')

    # ── CALCOLO FEATURE HL/LL ─────────────────────
    img_tensor_b = img_tensor.unsqueeze(0).to(DEVICE)
    feats_hl = extractor_hl(img_tensor_b)[0]
    feats_ll = extractor_ll(img_tensor_b)[0]

    if memory_bank_hl.device.type == "cpu":
        feats_hl = feats_hl.cpu()
        feats_ll = feats_ll.cpu()

    patch_scores_hl, _ = compute_anomaly_scores(feats_hl, memory_bank_hl)
    patch_scores_ll, _ = compute_anomaly_scores(feats_ll, memory_bank_ll)

    # ── FUSIONE HEATMAP HL+LL ──────────────────────
    hmap_fused = fuse_patch_heatmaps_max( patch_scores_ll.numpy(),patch_scores_hl.numpy(),IMG_SIZE)
    overlay_fused = overlay_heatmap(img_pil, hmap_fused)

    # ── CALCOLO AP SE MASK ESISTE ─────────────────
    if os.path.exists(gt_mask_path):
        gt_resized = cv2.resize(gt_mask, (hmap_fused.shape[1], hmap_fused.shape[0]), interpolation=cv2.INTER_NEAREST)
        ap_fused = average_precision_score(gt_resized.flatten(), hmap_fused.flatten()) if gt_resized.sum()>0 else 0.0
    else:
        ap_fused = 0.0

    axs[row_base+1, col].imshow(overlay_fused)
    axs[row_base+1, col].set_title(f"Fused HL+LL\nAP: {ap_fused:.3f}")
    axs[row_base+1, col].axis('off')

# ── DISABILITA ASSI EXTRA ─────────────────────────
for r in range(n_rows):
    for c in range(IMGS_PER_ROW):
        if r*IMGS_PER_ROW + c >= n_images:
            axs[r, c].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=150)
print(f"Visualizzazione salvata in {OUTPUT_PATH}")
plt.show()

In [ ]:
import cv2
import numpy as np


def segment_grabcut(
    image_bgr,
    heatmap_raw,
    fg_thresh,
    sure_fg_thresh,
    img_size=224,
    iter_count=5
):
    """
    GrabCut-guided refinement della heatmap.

    OUTPUT:
        refined_heatmap : heatmap continua raffinata
        final_mask      : maschera binaria GrabCut
        gc_mask         : mask multiclass originale GrabCut
    """

    # ───────────────────────────────
    # HEATMAP CONTINUA (NON NORMALIZZATA)
    # ───────────────────────────────
    h = heatmap_raw.astype(np.float32)

    h_up = cv2.resize(
        h,
        (img_size, img_size),
        interpolation=cv2.INTER_CUBIC
    )

    # ───────────────────────────────
    # IMMAGINE
    # ───────────────────────────────
    image_resized = cv2.resize(
        image_bgr,
        (img_size, img_size),
        interpolation=cv2.INTER_LINEAR
    )

    # ───────────────────────────────
    # CHECK SEED FOREGROUND
    # ───────────────────────────────
    fg_pixels = np.sum(h_up > fg_thresh)
    sure_fg_pixels = np.sum(h_up > sure_fg_thresh)

    # ───────────────────────────────
    # FALLBACK: NO ANOMALY
    # ───────────────────────────────
    if fg_pixels == 0 and sure_fg_pixels == 0:

        final_mask = np.zeros(
            (img_size, img_size),
            dtype=np.uint8
        )

        # heatmap invariata
        refined_heatmap = h_up.copy()

        gc_mask = np.full(
            (img_size, img_size),
            cv2.GC_BGD,
            dtype=np.uint8
        )

        return refined_heatmap, final_mask, gc_mask

    # ───────────────────────────────
    # INIT GRABCUT MASK
    # ───────────────────────────────
    gc_mask = np.full(
        (img_size, img_size),
        cv2.GC_BGD,
        dtype=np.uint8
    )

    # probable foreground
    gc_mask[h_up > fg_thresh] = cv2.GC_PR_FGD

    # sure foreground
    gc_mask[h_up > sure_fg_thresh] = cv2.GC_FGD

    # ───────────────────────────────
    # GRABCUT
    # ───────────────────────────────
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    cv2.grabCut(
        image_resized,
        gc_mask,
        None,
        bgd_model,
        fgd_model,
        iterCount=iter_count,
        mode=cv2.GC_INIT_WITH_MASK
    )

    # ───────────────────────────────
    # MASK BINARIA
    # ───────────────────────────────
    final_mask = np.where(
        (gc_mask == cv2.GC_FGD) |
        (gc_mask == cv2.GC_PR_FGD),
        1,
        0
    ).astype(np.uint8)

    # ───────────────────────────────
    # PESI SOFT DA GRABCUT
    # ───────────────────────────────
    soft_weights = np.zeros_like(h_up, dtype=np.float32)

    # background sicuro
    soft_weights[gc_mask == cv2.GC_BGD] = 0.4

    # background probabile
    soft_weights[gc_mask == cv2.GC_PR_BGD] = 0.6

    # foreground probabile
    soft_weights[gc_mask == cv2.GC_PR_FGD] = 0.8

    # foreground sicuro
    soft_weights[gc_mask == cv2.GC_FGD] = 1.0

    # ───────────────────────────────
    # HEATMAP RAFFINATA SOFT
    # ───────────────────────────────
    refined_heatmap = h_up * soft_weights
    return refined_heatmap, final_mask, gc_mask

In [ ]:
import numpy as np


def fuse_heatmaps_max(hmap1, hmap2):
    """
    Fusione point-wise max tra due heatmap.

    Parameters
    ----------
    hmap1 : np.ndarray
    hmap2 : np.ndarray

    Returns
    -------
    fused : np.ndarray
        Heatmap risultante:
        fused[i,j] = max(hmap1[i,j], hmap2[i,j])
    """

    hmap1 = np.asarray(hmap1, dtype=np.float32)
    hmap2 = np.asarray(hmap2, dtype=np.float32)

    if hmap1.shape != hmap2.shape:
        raise ValueError(
            f"Shape incompatibili: {hmap1.shape} vs {hmap2.shape}"
        )

    fused = np.maximum(hmap1, hmap2)

    return fused

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import average_precision_score
from math import ceil
from torchvision import transforms

OUTPUT_FILE_NAME = "anomaly_visualization_fused_2rows.png"
OUTPUT_PATH = f"/content/drive/MyDrive/patchcore_output/{OUTPUT_FILE_NAME}"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

IMGS_PER_ROW = 5
IMG_SIZE_PLOT = 224
ALPHA = 0.5

FG_THRESH=0.7
SURE_FG_THRESH=0.8

def denormalize_tensor(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=img_tensor.device)[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225], device=img_tensor.device)[:, None, None]
    img = img_tensor * std + mean
    img = torch.clamp(img, 0, 1)
    img = (img * 255).byte()
    return transforms.ToPILImage()(img.cpu())

def overlay_heatmap(img_pil, hmap, alpha=ALPHA, colormap=cv2.COLORMAP_JET):
    heatmap_color = cv2.applyColorMap((hmap*255).astype(np.uint8), colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(np.array(img_pil.resize((IMG_SIZE_PLOT, IMG_SIZE_PLOT))), alpha, heatmap_color, 1-alpha, 0)
    return overlay

# ── PREPARAZIONE FIGURA ──────────────────────────
n_images = len(test_anom)
n_rows = ceil(n_images / IMGS_PER_ROW) * 2  # base + fused heatmap

fig, axs = plt.subplots(n_rows, IMGS_PER_ROW, figsize=(IMGS_PER_ROW*4, n_rows*4))
axs = np.array(axs)

for idx in range(n_images):
    img_tensor, label, path = test_anom[idx]
    img_pil = denormalize_tensor(img_tensor)
    img_to_plot = np.array(img_pil.resize((IMG_SIZE_PLOT, IMG_SIZE_PLOT)))

    row_base = (idx // IMGS_PER_ROW) * 2
    col = idx % IMGS_PER_ROW

    # ── IMMAGINE BASE + GT ─────────────────────────
    rel_dir = os.path.relpath(os.path.dirname(path), os.path.join(BASE_DIR, "train"))
    gt_mask_path = os.path.join(BASE_DIR, "ground_truth_train", rel_dir, os.path.basename(path))

    if os.path.exists(gt_mask_path):
        gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 0).astype(np.uint8)
        gt_mask_resized = cv2.resize(gt_mask, (IMG_SIZE_PLOT, IMG_SIZE_PLOT), interpolation=cv2.INTER_NEAREST)
        overlay_gt = img_to_plot.copy()
        overlay_gt[gt_mask_resized==1] = [255, 0, 0]
        img_base = cv2.addWeighted(img_to_plot, 0.7, overlay_gt, 0.3, 0)
    else:
        img_base = img_to_plot

    axs[row_base, col].imshow(img_base)
    axs[row_base, col].set_title(f"GT {idx}")
    axs[row_base, col].axis('off')

    # ── CALCOLO FEATURE HL/LL ─────────────────────
    img_tensor_b = img_tensor.unsqueeze(0).to(DEVICE)
    feats_hl = extractor_hl(img_tensor_b)[0]
    feats_ll = extractor_ll(img_tensor_b)[0]

    if memory_bank_hl.device.type == "cpu":
        feats_hl = feats_hl.cpu()
        feats_ll = feats_ll.cpu()

    patch_scores_hl, _ = compute_anomaly_scores(feats_hl, memory_bank_hl)
    patch_scores_ll, _ = compute_anomaly_scores(feats_ll, memory_bank_ll)

    hmap1 = patch_scores_to_heatmap(patch_scores_hl, IMG_SIZE_PLOT)
    hmap2 = patch_scores_to_heatmap(patch_scores_ll, IMG_SIZE_PLOT)

    # ── FUSIONE HEATMAP HL+LL ──────────────────────
    #hmap_fused = fuse_patch_heatmaps_max( patch_scores_ll.numpy(),patch_scores_hl.numpy(),IMG_SIZE)
    hmap1,_,_ = segment_grabcut(
    image_bgr=cv2.cvtColor(img_to_plot.astype(np.uint8), cv2.COLOR_RGB2BGR),
    heatmap_raw=hmap1,
    fg_thresh=FG_THRESH,          # <-- scegli tu
    sure_fg_thresh=SURE_FG_THRESH # <-- scegli tu
    )
    hmap2,_,_ = segment_grabcut(
    image_bgr=cv2.cvtColor(img_to_plot.astype(np.uint8), cv2.COLOR_RGB2BGR),
    heatmap_raw=hmap2,
    fg_thresh=FG_THRESH,          # <-- scegli tu
    sure_fg_thresh=SURE_FG_THRESH # <-- scegli tu
    )
    hmap_fused=fuse_heatmaps_max(hmap1,hmap2)
    overlay_fused = overlay_heatmap(img_pil, hmap_fused)

    # ── CALCOLO AP SE MASK ESISTE ─────────────────
    if os.path.exists(gt_mask_path):
        gt_resized = cv2.resize(gt_mask, (hmap_fused.shape[1], hmap_fused.shape[0]), interpolation=cv2.INTER_NEAREST)
        ap_fused = average_precision_score(gt_resized.flatten(), hmap_fused.flatten()) if gt_resized.sum()>0 else 0.0
    else:
        ap_fused = 0.0

    axs[row_base+1, col].imshow(overlay_fused)
    axs[row_base+1, col].set_title(f"Fused HL+LL\nAP: {ap_fused:.3f}")
    axs[row_base+1, col].axis('off')

# ── DISABILITA ASSI EXTRA ─────────────────────────
for r in range(n_rows):
    for c in range(IMGS_PER_ROW):
        if r*IMGS_PER_ROW + c >= n_images:
            axs[r, c].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=150)
print(f"Visualizzazione salvata in {OUTPUT_PATH}")
plt.show()